## Part 1: Setup & Document Selection 

In [2]:
pip install python-dotenv langsmith openai

Note: you may need to restart the kernel to use updated packages.


In [14]:
from pathlib import Path
import os
from dotenv import load_dotenv

# Load the project .env regardless of the notebook server's working directory.
project_dir = Path.cwd()
env_file = next((path / '.env' for path in [project_dir, *project_dir.parents] if (path / '.env').exists()), project_dir / '.env')
load_dotenv(env_file)

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_PROJECT'] = 'langsmith-custom-dataset-evaluation'

if not os.getenv('OPENAI_API_KEY') or not os.getenv('LANGSMITH_API_KEY'):
    raise RuntimeError(f'Missing API keys. Expected them in {env_file.resolve()}')

print('Environment loaded from:', env_file.resolve())
print('OpenAI key loaded:', bool(os.getenv('OPENAI_API_KEY')))
print('LangSmith key loaded:', bool(os.getenv('LANGSMITH_API_KEY')))


OpenAI key loaded: True
LangSmith key loaded: True
OpenAI key loaded: True


In [20]:
# Create the wrapped client
from openai import OpenAI
from langsmith.wrappers import wrap_openai

client = wrap_openai(OpenAI())

# This call is a connectivity/tracing smoke test.
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Reply with the word ready.'}]
)
print(response.choices[0].message.content)

Your AI client has been successfully created! ✅


In [ ]:
from langsmith import traceable

@traceable
def target_function(inputs):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": inputs["question"]}
        ]
    )

    return {'answer': response.choices[0].message.content}

# Calling the decorated function creates a visible traced run.
test_result = target_function({'question': 'What is ArTee?'})
print(test_result)

In [ ]:
from langsmith import Client

ls_client = Client()
dataset_name = 'retwin-arte-evaluation-dataset'
dataset_description = 'Questions evaluating assistant understanding of Re-Twin and ArTee.'

examples = [
    {'question': 'What is Re-Twin?', 'answer': 'Re-Twin is a platform for battery energy storage system projects.'},
    {'question': 'What is ArTee?', 'answer': 'ArTee is a Re-Twin product that helps users understand and evaluate energy data.'},
    {'question': 'What kind of systems does Re-Twin support?', 'answer': 'It supports battery energy storage systems, also called BESS.'},
    {'question': 'How can ArTee help users make decisions?', 'answer': 'It helps users explore project scenarios, understand energy data, and make informed decisions.'},
    {'question': 'Can ArTee provide reports?', 'answer': 'Yes, ArTee can provide reports based on the evaluated project information.'},
    {'question': 'Why is energy data important in a BESS project?', 'answer': 'Energy data helps users assess project behavior, performance, and possible scenarios.'},
    {'question': 'Who might use ArTee?', 'answer': 'People evaluating or working with battery energy storage system projects may use ArTee.'},
    {'question': 'What does ArTee help users understand?', 'answer': 'It helps users understand energy data and the implications of BESS project scenarios.'},
    {'question': 'Does ArTee replace the entire BESS platform?', 'answer': 'No. ArTee is a product within the wider Re-Twin BESS platform.'},
    {'question': 'What is the main purpose of this evaluation dataset?', 'answer': 'It tests whether an assistant can explain Re-Twin, ArTee, and their role in BESS projects accurately and simply.'},
]

existing = list(ls_client.list_datasets(dataset_name=dataset_name, limit=1))
if existing:
    dataset = existing[0]
    print(f'Using existing dataset: {dataset.name}')
else:
    dataset = ls_client.create_dataset(dataset_name, description=dataset_description)
    ls_client.create_examples(
        dataset_id=dataset.id,
        inputs=[{'question': item['question']} for item in examples],
        outputs=[{'answer': item['answer']} for item in examples],
    )
    print(f'Created dataset: {dataset.name}')

print('Dataset ID:', dataset.id)

In [ ]:
def answer_is_nonempty(inputs, outputs):
    answer = outputs.get('answer', '') if isinstance(outputs, dict) else str(outputs)
    return {'key': 'answer_is_nonempty', 'score': int(bool(str(answer).strip()))}

evaluation = ls_client.evaluate(
    target_function,
    data=dataset.name,
    evaluators=[answer_is_nonempty],
    experiment_prefix='arte-gpt4o-mini',
    description='Evaluation of the traced ArTee target function.',
)

print('Evaluation submitted to LangSmith.')
print('Open the project:', os.environ['LANGSMITH_PROJECT'])